In [83]:
from utils import *
from utils.new_custom_classes import ParameterField
from utils.sigmav_functions import *
import numpy as np
import matplotlib.pyplot as plt
import itertools
import plotly.graph_objects as go
from tqdm import tqdm
from scipy.integrate import solve_ivp

import pandas as pd
import seaborn as sns

# Parametrization points

In [ ]:
# generic number of values for each parameter used for the parametric analysis
param_points = 3

# OPTIONAL - if not specified, the time necessary to reach a 50%D-50%T concentration in the plasma will be calculated
# total time for the parametric analysis [s]
total_time = 10* u.yr

# Parameters

### Plasma parameters

In [85]:
# Basic parameter fields (no time or spatial dependence)
V_plasma_field = ParameterField(
    parametrization_type="normal", mean=150, std=15, unit=u.m**3, 
    param_points=1, name="plasma_volume"
)

tau_p_T_field = ParameterField(
    parametrization_type="linear", min_val = 0.1, max_val=5, unit=u.s,
    param_points=3, name="tau_p_T"
)

tau_p_He3_field = ParameterField(
    parametrization_type="normal", mean=1, std=0.5, unit=u.s,
    param_points=1, name="tau_p_He3"
)


P_aux_field = ParameterField(
    parametrization_type="linear", min_val=20, max_val=100, unit=u.MW,
    param_points=1, name="P_aux"
)

P_lost_rad_field = ParameterField(
    parametrization_type="linear", min_val=0, max_val=20, unit=u.MW,
    param_points=param_points, name="P_lost_rad"
)

P_aux_all_DT_field = ParameterField(
    parametrization_type="linear", min_val=20, max_val=100, unit=u.MW,
    param_points=1, name="P_aux"
)

P_lost_rad_all_DT_field = ParameterField(
    parametrization_type="linear", min_val=0, max_val=20, unit=u.MW,
    param_points=1, name="P_lost_rad"
)

print("Basic parameter fields created:")
print(f"V_plasma: {V_plasma_field}")
print(f"tau_p_T: {tau_p_T_field}")
print(f"tau_p_He3: {tau_p_He3_field}")
print(f"P_aux: {P_aux_field}")
print(f"P_aux_all_DT: {P_aux_all_DT_field}")
print(f"P_lost_rad: {P_lost_rad_field}")
print(f"P_lost_rad_all_DT: {P_lost_rad_all_DT_field}")

Basic parameter fields created:
V_plasma: <ParameterField 'plasma_volume' type=normal, profile=none, shape=(1,), unit=meter ** 3>
[150.000] meter ** 3
tau_p_T: <ParameterField 'tau_p_T' type=linear, profile=none, shape=(3,), unit=second>
[0.100 2.550 5.000] second
tau_p_He3: <ParameterField 'tau_p_He3' type=normal, profile=none, shape=(1,), unit=second>
[1.000] second
P_aux: <ParameterField 'P_aux' type=linear, profile=none, shape=(1,), unit=megawatt>
[60.000] megawatt
P_aux_all_DT: <ParameterField 'P_aux' type=linear, profile=none, shape=(1,), unit=megawatt>
[60.000] megawatt
P_lost_rad: <ParameterField 'P_lost_rad' type=linear, profile=none, shape=(3,), unit=megawatt>
[0.000 10.000 20.000] megawatt
P_lost_rad_all_DT: <ParameterField 'P_lost_rad' type=linear, profile=none, shape=(1,), unit=megawatt>
[10.000] megawatt


##### Spataial parameters (pedestal profile)

In [86]:
# Create spatial parameter fields for temperature and density
T_i_spatial = ParameterField(
    parametrization_type="linear",
    spatial_profile="pedestal",
    min_val=14, max_val=20, unit=u.keV,
    param_points=param_points,
    space_points=space_points,
    value_center=1.0,        # Relative to mean
    value_ped=0.22,          # 5.5 keV / 25 keV ≈ 0.22
    value_edge=0.0004,       # 0.01 keV / 25 keV ≈ 0.0004
    transition_ratio=0.95,
    name="T_i_spatial"
)

n_tot_spatial = ParameterField(
    parametrization_type="linear",
    spatial_profile="pedestal", 
    min_val=1.3e20, max_val=2.1e20, unit=u.m**(-3),
    param_points=param_points,
    space_points=space_points,
    value_center=1.0,        # Relative to mean
    value_ped=0.67,          # 1.45e20 / 2.15e20 ≈ 0.67
    value_edge=0.047,        # 1e19 / 2.15e20 ≈ 0.047
    transition_ratio=0.95,
    name="n_tot_spatial"
)

print("Spatial parameter fields:")
print(f"T_i_spatial: {T_i_spatial}")
print(f"n_tot_spatial: {n_tot_spatial}")


Spatial parameter fields:
T_i_spatial: <ParameterField 'T_i_spatial' type=linear, profile=pedestal, shape=(3,), unit=kiloelectron_volt>
[14.000 17.000 20.000] kiloelectron_volt
n_tot_spatial: <ParameterField 'n_tot_spatial' type=linear, profile=pedestal, shape=(3,), unit=1 / meter ** 3>
[130000000000000000000.000 170000000000000000000.000 210000000000000000000.000] 1 / meter ** 3


##### Fuel Injection function

### Breeding and inventory parameters

In [87]:
TBR_DT_field = ParameterField(
    parametrization_type="linear", min_val=1.05, max_val=1.15,
    param_points=2, name="TBR_DT"
)

TBR_DDn_field = ParameterField(
    parametrization_type="linear", min_val=0.5, max_val=0.9,
    param_points=3, name="TBR_DDn"
)

tau_ifc_field = ParameterField(
    parametrization_type="linear", min_val=1, max_val=12, unit=u.h,
    param_points=3, name="tau_ifc"
)

tau_ofc_field = ParameterField(
    parametrization_type="linear", min_val=1, max_val=24, unit=u.h,
    param_points=3, name="tau_ofc"
)



print("Breeding parameters:")
print(f"TBR_DT: {TBR_DT_field}")
print(f"TBR_DDn: {TBR_DDn_field}")
print(f"tau_ifc: {tau_ifc_field}")
print(f"tau_ofc: {tau_ofc_field}")

Breeding parameters:
TBR_DT: <ParameterField 'TBR_DT' type=linear, profile=none, shape=(2,), unit=dimensionless>
[1.050 1.150] dimensionless
TBR_DDn: <ParameterField 'TBR_DDn' type=linear, profile=none, shape=(3,), unit=dimensionless>
[0.500 0.700 0.900] dimensionless
tau_ifc: <ParameterField 'tau_ifc' type=linear, profile=none, shape=(3,), unit=hour>
[1.000 6.500 12.000] hour
tau_ofc: <ParameterField 'tau_ofc' type=linear, profile=none, shape=(3,), unit=hour>
[1.000 12.500 24.000] hour


### Energy production parameters

In [88]:
# Economic parameters
eta_th_field = ParameterField(
    parametrization_type="linear", min_val=0.3, max_val=0.4,
    param_points=2, name="eta_th"
)

plant_avail_field = ParameterField(
    parametrization_type="linear", min_val=0.5, max_val=0.9,
    param_points=5, name="plant_availability"
)

Cost_per_kWh_field = ParameterField(
    parametrization_type="normal", mean=0.25, std=0.15, unit=1/u.kWh,
    param_points=5, name="Cost_per_kWh"
)

print("Economic parameters:")
print(f"eta_th: {eta_th_field}")
print(f"plant_avail: {plant_avail_field}")
print(f"Cost_per_kWh: {Cost_per_kWh_field}")

Economic parameters:
eta_th: <ParameterField 'eta_th' type=linear, profile=none, shape=(2,), unit=dimensionless>
[0.300 0.400] dimensionless
plant_avail: <ParameterField 'plant_availability' type=linear, profile=none, shape=(5,), unit=dimensionless>
[0.500 0.600 0.700 0.800 0.900] dimensionless
Cost_per_kWh: <ParameterField 'Cost_per_kWh' type=normal, profile=none, shape=(5,), unit=1 / kilowatt_hour>
[0.105 0.185 0.250 0.315 0.395] 1 / kilowatt_hour


# Perform the parametric analysis

In [89]:
input_data = [
    V_plasma_field.data,
    tau_p_T_field.data, 
    tau_p_He3_field.data,
    P_aux_field.data,
    P_lost_rad_field.data,
    P_aux_all_DT_field.data,
    P_lost_rad_all_DT_field.data,
    
    T_i_spatial.data,
    n_tot_spatial.data,
    
    TBR_DT_field.data,
    TBR_DDn_field.data,
    tau_ifc_field.data,
    tau_ofc_field.data,
    
    eta_th_field.data,
    plant_avail_field.data,
    Cost_per_kWh_field.data,
]


In [ ]:
# Create iterator based only on the number of parameter variations (first dimension)
param_ranges = [range(data.shape[0]) for data in input_data]

results = []


def tritium_inventory_odes(t,y):
    N_ofc = y[0] # total number of tritium atoms in the outer fuel cycle
    N_ifc = y[1] # total number of tritium atoms in the inner fuel cycle
    N_st = y[2]  # total number of tritium atoms in the storage
    n_T = y[3]  # total tritium density in the plasma (float)
    
    n_spatial = n_tot.size  # or set explicitly
    n_T = n_T/u.m**3
    
    if n_spatial == 1:
        # Single spatial point - no integration needed
        n_D = n_tot - n_T
        Tdot_DDn = (TBR_DDn*0.5*n_D**2*sigmav_DD_n*V_plasma).to('1/s')
        Tdot_DDp = (0.5*n_D**2*sigmav_DD_p*V_plasma).to('1/s')
        Tdot_DT = (TBR_DT*n_D*n_T*sigmav_DT*V_plasma).to('1/s')
        Tdot_burn = (n_D*n_T*sigmav_DT*V_plasma).to('1/s')
        # set injection rate so that satisfies these constraints:
        N_st_min = 0.001/tritium_mass.to('kg').magnitude
        # - if N_st < N_st_min, injection_rate = 0
        # - else, injection_rate = N_ifc/tau_ifc - lambda_T*N_st (tries to inject all the T that enters the storage)
        # - maximum injection rate is injection_rate_max = n_T50/tau_p_T*V_plasma + Tdot_burn5050 - T_dot_DDp5050 (the injection rate necessary to maintain a 50D50T plasma)
        injection_rate_max = (n_tot/2/tau_p_T*V_plasma + 0.25*n_tot**2*sigmav_DT*V_plasma - 0.25/2*n_tot**2*sigmav_DD_p*V_plasma).to('1/s')
        if N_st < N_st_min:
            injection_rate = 0 * u.s**(-1)
        else:
            injection_rate = min((N_ifc/tau_ifc - lambda_T*N_st), injection_rate_max).to('1/s')
    elif n_spatial > 1:
        # Multiple spatial points - integrate over space
        n_tot_integrated = np.trapz(n_tot, x=np.linspace(0, 1, n_spatial))
        n_D = n_tot_integrated - n_T
        Tdot_DDn = np.trapz(TBR_DDn*0.5*n_D**2*sigmav_DD_n*V_plasma, x=np.linspace(0, 1, n_spatial)).to('1/s')
        Tdot_DDp = np.trapz(0.5*n_D**2*sigmav_DD_p*V_plasma, x=np.linspace(0, 1, n_spatial)).to('1/s')
        Tdot_DT = np.trapz(TBR_DT*n_D*n_T*sigmav_DT*V_plasma, x=np.linspace(0, 1, n_spatial)).to('1/s')
        Tdot_burn = np.trapz(n_D*n_T*sigmav_DT*V_plasma, x=np.linspace(0, 1, n_spatial)).to('1/s')
        # set injection rate so that satisfies these constraints:
        N_st_min = 0.001/tritium_mass.to('kg').magnitude
        # - if N_st < N_st_min, injection_rate = 0
        # - else, injection_rate = N_ifc/tau_ifc - lambda_T*N_st (tries to inject all the T that enters the storage)
        # - maximum injection rate is injection_rate_max = n_T50/tau_p_T*V_plasma + Tdot_burn5050 - T_dot_DDp5050 (the injection rate necessary to maintain a 50D50T plasma)
        injection_rate_max = np.trapz((n_tot/2/tau_p_T*V_plasma + 0.25*n_tot**2*sigmav_DT*V_plasma - 0.25/2*n_tot**2*sigmav_DD_p*V_plasma), x=np.linspace(0, 1, n_spatial)).to('1/s')
        if N_st < N_st_min:
            injection_rate = np.zeros_like(n_tot) * u.s**(-1)
        else:
            injection_rate = min(np.trapz(N_ifc/tau_ifc - lambda_T*N_st, x=np.linspace(0, 1, n_spatial))*np.ones_like(n_tot), injection_rate_max).to('1/s')
    
   
    dN_ofc_dt = (Tdot_DT + Tdot_DDn - N_ofc / tau_ofc - N_ofc*lambda_T).to('1/s')
    dN_ifc_dt = (N_ofc / tau_ofc - N_ifc / tau_ifc  - lambda_T * N_ifc + n_T/tau_p_T*V_plasma).to('1/s')
    dN_stor_dt = (N_ifc / tau_ifc - lambda_T * N_st - injection_rate).to('1/s')
    dnT_dt = (injection_rate/V_plasma + Tdot_DDp/V_plasma - n_T/tau_p_T - Tdot_burn/V_plasma).to('1/s/m^3')

    total_T_produced = (Tdot_DDn + Tdot_DDp + Tdot_DT).to('1/s')
    total_T_burnt = (n_D*n_T*sigmav_DT*V_plasma).to('1/s')

    net_T_produced = total_T_produced - total_T_burnt

    return [float(dN_ofc_dt.to('1/s').magnitude), float(dN_ifc_dt.to('1/s').magnitude), float(dN_stor_dt.to('1/s').magnitude), float(dnT_dt.to('1/s/m^3').magnitude)]

    
def DT_reached(t, y):
    # Get tritium profile from y
    n_spatial = n_tot.size
    n_T = np.array(y[3:3+n_spatial])
    # Integrate tritium over space (use sum or trapz if you have spatial grid)
    total_tritium = np.sum(n_T)
    # Integrate total inventory over space
    total_n_tot = np.sum(n_tot.magnitude)
    # Stop when total tritium reaches 50% of total inventory
    return total_tritium - 0.5 * total_n_tot

def NEGATIVE(t, y):
    # stop when y[0] or y[1] or y[2] or y[3] is negative
    return min(y[0]+1e-10, y[1]+1e-10, y[2]+1e-10, y[3]+1e-10)

# Time span
t_span = (0, total_time.to('s').magnitude)
t_eval = np.linspace(*t_span, 1000)
    
for param_combo in tqdm(itertools.product(*param_ranges), 
                       total=np.prod([data.shape[0] for data in input_data]), 
                       desc="Parametric analysis"):
    
    # Extract data for each parameter
    extracted_data = [input_data[i][param_idx] for i, param_idx in enumerate(param_combo)]
        
    # Unpack the extracted data
    (V_plasma, tau_p_T, tau_p_He3, P_aux, P_lost_rad, P_aux_all_DT, P_lost_rad_all_DT,
     T_i, n_tot,
     
     TBR_DT, TBR_DDn, tau_ifc, tau_ofc,
     
     eta_th, plant_avail, Cost_per_kWh,
     ) = extracted_data


    # Get cross-sections
    sigmav_DD = sigmav_DD_BoschHale(T_i)[0].to('m^3/s')  # [m^3/s]
    sigmav_DD_p = sigmav_DD_BoschHale(T_i)[1].to('m^3/s')  # [m^3/s]
    sigmav_DD_n = sigmav_DD_BoschHale(T_i)[2].to('m^3/s')  # [m^3/s]
    sigmav_DT = sigmav_DT_BoschHale(T_i).to('m^3/s')    # [m^3/s]
    sigmav_DHe3 = sigmav_DHe3_BoschHale(T_i).to('m^3/s')   # [m^3/s]
    
    y0 = np.zeros(3 + n_tot.size)  # Initial conditions: [N_ofc, N_ifc, N_st, n_T]

    # stop whe  DT reached
    DT_reached.terminal = True
    NEGATIVE.terminal = True
    # Solve ODE
    
    
    sol = solve_ivp(
        fun = tritium_inventory_odes, 
        t_span = t_span, 
        t_eval = t_eval,
        y0 = y0, 
        method = 'BDF', 
        dense_output=False,
        events = [DT_reached, NEGATIVE], )
    
    if sol.t_events[1].size > 0:
        print(f"Negative event occurred at t={sol.t_events[1]}")
        break
    elif sol.t_events[0].size > 0:
        t_startup = sol.t_events[0][0]*u.s
    else:
        t_startup = np.inf*u.s
        
    # calculate the energy produced
    n_T = sol.y[3] * u.m**(-3)  # shape (n_spatial, n_time)
    if n_tot.size == 1:
        n_D = n_tot - n_T
        P_DDn = n_D*n_D*sigmav_DD_n/2*V_plasma*E_DDn
        P_DDp = n_D*n_D*sigmav_DD_p/2*V_plasma*E_DDp
        P_DT = n_D*n_T*sigmav_DT*V_plasma*E_DT
        # equivalent energy if always DT
        P_DT_full = n_tot/2*n_tot/2*sigmav_DT*V_plasma*E_DT
    else:
        n_D = np.trapz(n_tot, x=np.linspace(0, 1, n_tot.size)) - n_T
        P_DDn = np.trapz(n_D*n_D*sigmav_DD_n/2*V_plasma, x=np.linspace(0, 1, n_tot.size))*u.W
        P_DDp = np.trapz(n_D*n_D*sigmav_DD_p/2*V_plasma, x=np.linspace(0, 1, n_tot.size))*u.W
        P_DT = np.trapz(n_D*n_T*sigmav_DT*V_plasma, x=np.linspace(0, 1, n_tot.size))*u.W
        # equivalent energy if always DT
        P_DT_full = np.trapz(n_tot/2*n_tot/2*sigmav_DT*V_plasma, x=np.linspace(0, 1, n_tot.size))*u.W


    
    # energy lost
    # Integrate over time (up to t_startup if not infinite, else total_time)
    t_end = t_startup.to('s').magnitude if np.isfinite(t_startup.to('s').magnitude) else total_time.to('s').magnitude
    mask = sol.t <= t_end

    net_energy_DD = np.trapz((P_DDn[mask]+P_DDp[mask]+P_DT[mask]-P_lost_rad*np.ones(sol.t[mask].shape))-P_aux*np.ones(sol.t[mask].shape), sol.t[mask]*u.s)
    net_energy_DT_full = np.trapz((P_DT_full-P_lost_rad_all_DT)*np.ones(sol.t[mask].shape)-P_aux_all_DT*np.ones(sol.t[mask].shape), sol.t[mask]*u.s)

    E_lost = eta_th*(net_energy_DT_full-net_energy_DD)

    Dollar_Lost = E_lost * Cost_per_kWh
    
    n_T = sol.y[3] * u.m**(-3)
    I_ifc = sol.y[0] * tritium_mass.to('kg')
    I_ofc = sol.y[1] * tritium_mass.to('kg')
    I_stor = sol.y[2] * tritium_mass.to('kg')
    row = [
        # INPUTS
        V_plasma.to('m^3').magnitude,                     # 0
        tau_p_T.to('s').magnitude,                      # 1
        tau_p_He3.to('s').magnitude,                    # 2
        P_aux.to('MW').magnitude,               # 3
        P_aux_all_DT.to('MW').magnitude,        # 4
        P_lost_rad.to('MW').magnitude,          # 5
        P_lost_rad_all_DT.to('MW').magnitude,   # 6
        T_i.to('keV').magnitude,                          # 7
        n_tot.to('m^-3').magnitude,                        # 8
        #injection_rate_max.magnitude,           # -
        TBR_DT.magnitude if hasattr(TBR_DT, "magnitude") else TBR_DT,  # 9
        TBR_DDn.magnitude if hasattr(TBR_DDn, "magnitude") else TBR_DDn,  # 10
        tau_ifc.magnitude if hasattr(tau_ifc, "magnitude") else tau_ifc,  # 11
        tau_ofc.magnitude if hasattr(tau_ofc, "magnitude") else tau_ofc,  # 12
        eta_th.magnitude if hasattr(eta_th, "magnitude") else eta_th,  # 13
        plant_avail.magnitude if hasattr(plant_avail, "magnitude") else plant_avail,  # 14
        Cost_per_kWh.to('1/kWh').magnitude,                 # 15
        # OUTPUTS
        P_DT.to('MW'),                          # 16
        P_DDn.to('MW'),                         # 17
        P_DDp.to('MW'),                         # 18
        P_DT_full.to('MW'),                     # 19
        t_startup.to('hour'),         # 20
        E_lost.to('MJ'),              # 21
        Dollar_Lost.to(''),                  # 22
        n_T,                                    # 23
        I_ifc,                                  # 24
        I_ofc,                                  # 25
        I_stor                                  # 26
    ]
    results.append(row)

Parametric analysis:   0%|          | 0/218700 [00:00<?, ?it/s]

Parametric analysis:   0%|          | 23/218700 [00:20<53:34:05,  1.13it/s]

# save to csv

In [ ]:
# save results to a DataFrame
columns = [
    # INPUTS
    "V_plasma (m^3)",                     # 0
    "tau_p_T (s)",                        # 1
    "tau_p_He3 (s)",                      # 2
    "P_aux (MW)",                         # 3
    "P_aux_all_DT (MW)",                  # 4
    "P_lost_rad (MW)",                    # 5
    "P_lost_rad_all_DT (MW)",             # 6
    "T_i (keV)",                          # 7   
    "n_tot (m^-3)",                       # 8
    #"injection_rate_max",                 # -
    "TBR_DT",                             # 9
    "TBR_DDn",                            # 10
    "tau_ifc (h)",                        # 11
    "tau_ofc (h)",                        # 12  
    "eta_th",                             # 13
    "plant_avail",                        # 14
    "Cost_per_kWh (1/kWh)",               # 15
    # OUTPUTS
    "P_DT (MW)",                          # 16
    "P_DDn (MW)",                         # 17
    "P_DDp (MW)",                         # 18  
    "P_DT_full (MW)",                     # 19
    "t_startup (h)",                      # 20
    "E_lost (MJ)",                        # 21
    "Dollar_Lost ($)",                    # 22
    "n_T (m^-3)",                         # 23
    "I_ifc (kg)",                         # 24
    "I_ofc (kg)",                         # 25
    "I_stor (kg)"                         # 26
]

df_results = pd.DataFrame(results, columns=columns)
df_results.to_csv("parametric_study_results.csv", index=False)